# Phase 2: Probability and Distributions in the NACC UDS + MRI Subset

This notebook accompanies `src/phase2_analysis.py` and the Phase 2 report. It follows the NACC Track C requirements in `amarpr.pdf`, pages 8–10. It is written in English and uses the same participant-level definitions and data rules as the reproducible script.

**Research question:** Among supplied linked participants aged 65 or older, how do empirical probabilities of UDS-coded MCI or dementia differ across MRI-defined entorhinal-volume groups, and what do the MoCA and volume distributions and their linear association show?

The analysis is descriptive. It does not use a formal independence test, confidence intervals, hypothesis tests, causal estimates, or a population-representative sampling design.

## Track C checklist and data limits

The analysis defines participant-level events and a Bernoulli variable; gives marginal, joint, and conditional probabilities; compares MRI-defined groups; discusses empirical independence; describes a discrete distribution and hypothetical binomial count; examines both MoCA and MRI volume distributions with normal-assessment plots; calculates covariance and correlation; and checks Bayes' identity.

The supplied files contain one row per unique `NACCID`, but no visit identifiers/dates or MRI scan dates. Therefore, the files support a transparent rule of one supplied record per unique participant, but cannot identify true NACC visit counts, the number of visits per participant, a verified baseline, or UDS–MRI contemporaneity. Do not interpret the one supplied row as proof that each participant had exactly one visit. Calculations treat each unique participant as one observation.

In [ ]:
from pathlib import Path
import csv
import json
import math
import statistics
import subprocess
import sys
from collections import Counter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "udslong_imputed.csv").exists())
# Rebuild results, audit CSVs, and figures from the supplied CSVs.
subprocess.run([sys.executable, str(ROOT / "src" / "phase2_analysis.py")],
               check=True, cwd=ROOT)
RESULTS = json.loads((ROOT / "reports/phase2/phase2_results.json").read_text(encoding="utf-8"))
print("Project root:", ROOT)
print("Analysis output rebuilt; result sections:", ", ".join(RESULTS.keys()))

## Variables, linkage, and participant flow

The input files are `data/udslong_imputed.csv` (UDS) and `data/mriscanclean.csv` (MRI), joined by `NACCID`. The supplied one-row-per-ID records are checked for uniqueness and UDS/MRI `NACCUDSD` agreement before applying `NACCAGE >= 65`.

UDS variables relevant here include `NACCAGE`, `MOCATOTS`, and `NACCUDSD`. `NACCUDSD` codes are 1=Normal cognition, 2=Impaired not MCI, 3=MCI, and 4=Dementia. MoCA values outside [0, 30] are invalid; fractional in-range values are retained. MRI variables `LH_ENTORHINAL_GVOL` and `RH_ENTORHINAL_GVOL` are structural regional volumes, interpreted in mm³. Bilateral volume is their arithmetic mean when both sides are positive. It is an imaging measure, not a cognitive score.

The rows-per-ID counts below describe the supplied files only. They do not recover actual visit/encounter counts. The UDS file is processed/imputed; the original imputation procedure and missingness mask are unavailable.

In [ ]:
def read_rows(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

uds = read_rows(ROOT / "data/udslong_imputed.csv")
mri = read_rows(ROOT / "data/mriscanclean.csv")
ids_uds = [r["NACCID"] for r in uds]
ids_mri = [r["NACCID"] for r in mri]
assert len(ids_uds) == len(set(ids_uds))
assert len(ids_mri) == len(set(ids_mri))
u_by_id = {r["NACCID"]: r for r in uds}
m_by_id = {r["NACCID"]: r for r in mri}
matched = set(u_by_id) & set(m_by_id)
assert len(matched) == 1237
assert not (set(u_by_id) - set(m_by_id))
assert not (set(m_by_id) - set(u_by_id))
status_disagreements = sum(float(u_by_id[k]["NACCUDSD"]) != float(m_by_id[k]["NACCUDSD"]) for k in matched)
assert status_disagreements == 0
age65_ids = {k for k in matched if float(u_by_id[k]["NACCAGE"]) >= 65}
assert len(age65_ids) == 864

flow = RESULTS["flow"]
print("UDS and MRI records per supplied NACCID:",
      Counter(Counter(ids_uds).values()), Counter(Counter(ids_mri).values()))
print("Matched IDs:", len(matched), "| unmatched UDS:", len(set(u_by_id)-set(m_by_id)),
      "| unmatched MRI:", len(set(m_by_id)-set(u_by_id)))
print("Status-code disagreements:", status_disagreements)
print("\nAnalytic flow:")
for row in flow:
    print(f"- {row['step']}: {row['n']:,}")
print("Actual visits per participant: not recoverable from these files.")

## Event and random-variable definitions

For participant record \(r\), define:

- \(A(r)=1\) if `NACCUDSD` is 3 or 4 (the supplied UDS classification is MCI or dementia), and 0 otherwise. This is not a new clinical diagnosis.
- \(B(r)=1\) if bilateral entorhinal volume is at or below the age-eligible cohort median; otherwise \(B(r)=0\).
- \(X(r)=\mathbf{1}\{A(r)=1\}\), a Bernoulli participant-level variable.

The median uses all age-eligible participants with valid bilateral volume. It is linearly interpolated; ties at the median are assigned to \(B=1\). All probabilities in the 2×2 system use the same valid-status/valid-volume subset. The empirical population represented is the supplied linked NACC clinical/research subset aged 65+, not older adults in general.

In [ ]:
events = RESULTS["events"]
ct = RESULTS["contingency"]
prob = RESULTS["probabilities"]
print("A:", events["A"])
print("B:", events["B"])
print("Median (mm³):", events["median_mm3"])
print("Eligible participants with valid volume:", events["volume_median_n"])
print("Strictly below median:", events["below_median_n"])
print("Equal to median (ties in B=1):", events["equal_to_median_n"])
print("At or below median:", events["at_or_below_median_n"])
print("Common probability denominator:", ct["denominator"])

## Joint table, marginal probabilities, and conditional probabilities

For the common denominator \(n\), compute \(P(A)\), \(P(B)\), and \(P(A\cap B)\) from frequency counts. Then compare \(P(A\mid B)\), \(P(A\mid B^c)\), and \(P(B\mid A)\). The numerator and denominator are printed for each probability so the calculations can be audited.

In [ ]:
print("2×2 counts: rows=A, columns=B")
for row in ct["rows"]:
    print(f"A={row['A']:>3}, B={row['B']:>3}: {row['count']:>4} / {ct['denominator']}")
print("\nAuditable probability calculations:")
for name in ("P_A", "P_B", "P_A_and_B", "P_A_given_B", "P_A_given_not_B", "P_B_given_A"):
    item = prob[name]
    print(f"{name}: {item['numerator']}/{item['denominator']} = {item['value']:.6f}")
assert sum(row["count"] for row in ct["rows"]) == ct["denominator"]
assert all(0 <= prob[k]["value"] <= 1 for k in
           ("P_A", "P_B", "P_A_and_B", "P_A_given_B", "P_A_given_not_B", "P_B_given_A"))
assert all(prob[k]["denominator"] > 0 for k in
           ("P_A", "P_B", "P_A_and_B", "P_A_given_B", "P_A_given_not_B", "P_B_given_A"))

## MRI-defined group comparison and exploratory independence discussion

The two groups are pre-defined by the eligible-cohort median of bilateral entorhinal volume, not selected after examining their cognitive-status probabilities. The comparison is the empirical proportion with \(A=1\) in each MRI-defined group. It is descriptive and does not imply an MRI effect or causation. Every age-eligible participant has a valid bilateral volume in these supplied rows, so a valid-MRI versus no-valid-MRI comparison cannot be made; the volume groups provide the MRI-defined comparison.

In [ ]:
groups = RESULTS["groups"]
for g in groups:
    print(f"{g['group']}: A=1 in {g['n_MCI_or_dementia']}/{g['n_group']} "
          f"= {g['percent']:.2f}% (denominator: {g['denominator_definition']})")
print("\nStatus-code 2 (Impaired, not MCI) count among age-eligible participants:",
      sum(float(u_by_id[k]["NACCUDSD"]) == 2 for k in age65_ids))
print("P(A ∩ B) =", f"{prob['P_A_and_B']['value']:.6f}")
print("P(A)P(B) =", f"{prob['P_A_times_P_B']:.6f}")
print("P(A|B) =", f"{prob['P_A_given_B']['value']:.6f}",
      "versus P(A) =", f"{prob['P_A']['value']:.6f}")
print("Empirical difference P(A∩B)-P(A)P(B) =",
      f"{prob['independence_difference']:.6f}")
print("This compares same-subset frequencies; it is not a formal independence test.")

![Empirical probability of MCI or dementia by bilateral entorhinal-volume group. Bars display percentages and counts.](../figures/phase2/group_probabilities.svg)

## Bernoulli distribution and hypothetical binomial count

The empirical PMF for \(X=\mathbf{1}\{A\}\) has two outcomes. Its expectation is \(E[X]=p\); its empirical distribution variance uses divisor \(n\), \(\operatorname{Var}(X)=p(1-p)\).

For a hypothetical fixed \(n=10\), let \(Y\) count A outcomes in ten independent Bernoulli draws with the same \(p\). Then \(Y\sim\mathrm{Binomial}(10,p)\), \(E[Y]=np\), and \(\operatorname{Var}(Y)=np(1-p)\). This model assumes fixed trial count, independent trials, constant event probability, and two outcomes. It models hypothetical independent draws; selecting distinct participants without replacement from this finite dataset is not exactly binomial. Repeated visits are not independent trials.

In [ ]:
print("Empirical PMF of X:")
for row in RESULTS["bernoulli"]["pmf"]:
    print(f"X={row['x']}: {row['label']}; count={row['count']}; "
          f"P(X={row['x']})={row['probability']:.6f}")
print("E[X] =", f"{RESULTS['bernoulli']['E_X']:.6f}")
print("Empirical distribution Var(X) =", f"{RESULTS['bernoulli']['Var_X_empirical']:.6f}")
binomial = RESULTS["binomial"]
print(f"\nBinomial model: n={binomial['n']}, p={binomial['p']:.6f}")
for row in binomial["pmf"]:
    print(f"P(Y={row['k']}) = {row['probability']:.8f}")
print("E[Y] =", f"{binomial['expected']:.6f}")
print("Var(Y) =", f"{binomial['variance']:.6f}")
assert abs(sum(row["probability"] for row in binomial["pmf"]) - 1) < 1e-12

![Hypothetical Binomial(10, p) probability mass function.](../figures/phase2/binomial_pmf.svg)

## Continuous measures and visual normal assessment

MoCA (`MOCATOTS`) is bounded and discrete (valid scores from 0 to 30); it is not literally continuous. Bilateral entorhinal volume is a positive quantitative measure in mm³. For each variable, the summary reports valid \(n\), mean, median, empirical variance using divisor \(n\), moment skewness, and the count outside the 1.5×IQR fences. Outliers are retained. Histograms with fitted-normal overlays and normal Q–Q plots support visual assessment; no normality-test p-value is used.

In [ ]:
continuous = RESULTS["continuous"]
for name, s in continuous.items():
    print(f"{name}: n={s['n']}, mean={s['mean']:.4f}, "
          f"variance (divisor n)={s['variance_population_divisor_n']:.4f}, "
          f"median={s['median']:.4f}, skewness={s['skewness_moment']:.4f}, "
          f"1.5×IQR fences=[{s['fence_low']:.4f}, {s['fence_high']:.4f}], "
          f"outliers={s['outliers_1_5_iqr']}")
print("MoCA is left-skewed, discrete, and concentrated toward its upper bound;")
print("the normal approximation is weak near the ceiling and in the tails.")
print("Volume is mildly right-skewed with upper-tail departures; a normal fit is")
print("at most a rough description of its central shape.")

![Valid MoCA scores: histogram with fitted-normal overlay.](../figures/phase2/moca_histogram.svg)

![MoCA normal Q–Q plot.](../figures/phase2/moca_qq.svg)

![Valid bilateral entorhinal volume: histogram with fitted-normal overlay.](../figures/phase2/volume_histogram.svg)

![Bilateral entorhinal volume normal Q–Q plot.](../figures/phase2/volume_qq.svg)

## MoCA–volume covariance and scatterplot

Use pairwise-complete age-eligible participants for both quantitative variables. The reported covariance is sample covariance with denominator \(n-1\), in score-points·mm³. Pearson correlation describes direction and strength of linear association; it is not causal and no formal inference is performed.

In [ ]:
association = RESULTS["association"]
print("Pairwise-complete n:", association["n"])
print("Sample covariance (score-points·mm³):",
      f"{association['sample_covariance_MoCA_points_times_mm3']:.6f}")
print("Pearson correlation:", f"{association['Pearson_r']:.6f}")
print("Interpretation:", association["interpretation"])

![Pairwise-complete MoCA and bilateral entorhinal volume scatterplot with descriptive fitted line.](../figures/phase2/moca_volume_scatter.svg)

## Bayes' theorem and final interpretation

The same event table gives \(P(B\mid A)=n_{11}/n_A\). Independently, Bayes' theorem gives \(P(A\mid B)P(B)/P(A)\); the values must agree. Here the prior is the marginal probability \(P(B)\) in this eligible dataset, updated after conditioning on A. This is an event-frequency identity, not validated individual diagnostic prediction.

In [ ]:
lhs = prob["P_B_given_A"]["value"]
rhs = prob["P_A_given_B"]["value"] * prob["P_B"]["value"] / prob["P_A"]["value"]
print(f"P(B|A) = {prob['P_B_given_A']['numerator']}/"
      f"{prob['P_B_given_A']['denominator']} = {lhs:.8f}")
print("P(A|B)P(B)/P(A) =", f"{rhs:.8f}")
print("Absolute difference =", f"{abs(lhs-rhs):.3g}")
assert abs(lhs - rhs) < 1e-12

## Limitations and Track C record-selection note

- The analytic population is the supplied age-65+ linked NACC subset, not all older adults.
- Each input has one supplied row per unique participant. Visit IDs/dates and MRI scan dates are unavailable; actual visit count/distribution, baseline status, and contemporaneous UDS–MRI alignment cannot be determined.
- The UDS file is processed/imputed, but the method and original missingness mask are unavailable. No blanks in the supplied file do not prove the original data had no missing observations.
- Every age-eligible participant has a valid bilateral entorhinal volume, so no MRI-absent group exists for comparison. The median-defined MRI groups support the required group comparison.
- UDS `NACCUDSD` is a supplied classification, and empirical contrasts do not establish causation or population-wide risk.

**Questions to defend:** Why use the median rule and include ties in the lower group? What assumptions make the hypothetical binomial model reasonable, and why is finite sampling without replacement different? Why can these files not support a visit-level or baseline-aligned analysis?